# Netflix User Behavior, Retention and Churn Analysis

## Project Goal

The goal of this project is to analyze Netflix user behavior patterns, identify churn signals, measure retention, and generate product recommendations based on behavioral analytics.

---

## Main Objectives

- Analyze viewing behavior
- Measure engagement
- Detect churn signals
- Segment users
- Understand retention patterns
- Generate product insights

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# PostgreSQL Connection

In [ ]:
username = "postgres"
password = "password"
host = "localhost"
port = "5432"
database = "netflix_project"

engine = create_engine(
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

In [ ]:
query = "SELECT * FROM viewing_activity LIMIT 5"

df = pd.read_sql(query, engine)

df.head()

# 1. Data Understanding

In [ ]:
query = "SELECT * FROM viewing_activity"

view_df = pd.read_sql(query, engine)

In [ ]:
view_df.head()

In [ ]:
view_df.shape

In [ ]:
view_df.columns

In [ ]:
view_df.info()

In [ ]:
view_df.describe(include="all")

In [ ]:
view_df.isnull().sum()

In [ ]:
view_df.duplicated().sum()

# 2. Data Cleaning and Preprocessing

In [ ]:
view_df['Start Time'] = pd.to_datetime(view_df['Start Time'])

In [ ]:
view_df.info()

In [ ]:
view_df['Duration'].head()

In [ ]:
view_df['Duration'] = pd.to_timedelta(view_df['Duration'])

In [ ]:
view_df['duration_minutes'] = (
    view_df['Duration'].dt.total_seconds() / 60
)

In [ ]:
view_df[['Duration', 'duration_minutes']].head()

## Date Features

In [ ]:
view_df['date'] = view_df['Start Time'].dt.date

view_df['year'] = view_df['Start Time'].dt.year

view_df['month'] = view_df['Start Time'].dt.month

view_df['day'] = view_df['Start Time'].dt.day

view_df['day_name'] = view_df['Start Time'].dt.day_name()

view_df['hour'] = view_df['Start Time'].dt.hour

In [ ]:
view_df.head()

In [ ]:
view_df['Profile Name'].nunique()

In [ ]:
view_df['Profile Name'].value_counts()

In [ ]:
view_df['Title'].value_counts().head(10)

In [ ]:
view_df['duration_minutes'].mean()

In [ ]:
view_df['Device Type'].value_counts()

# 3. Feature Engineering

## Session Creation

In [ ]:
view_df['session_id'] = (
    view_df['Profile Name'].astype(str)
    + "_"
    + view_df['date'].astype(str)
)

In [ ]:
view_df[['Profile Name', 'date', 'session_id']].head()

## User Engagement Features

In [ ]:
user_metrics = view_df.groupby('Profile Name').agg({

    'Title': 'count',

    'date': 'nunique',

    'session_id': 'nunique',

    'duration_minutes': 'mean'

}).reset_index()

In [ ]:
user_metrics.columns = [

    'profile_name',

    'total_watch_count',

    'active_days',

    'total_sessions',

    'avg_watch_duration'

]

In [ ]:
user_metrics.head()

In [ ]:
user_metrics['avg_watch_per_day'] = (
    user_metrics['total_watch_count']
    / user_metrics['active_days']
)

In [ ]:
total_watch_time = (
    view_df.groupby('Profile Name')['duration_minutes']
    .sum()
    .reset_index()
)

In [ ]:
total_watch_time.columns = [
    'profile_name',
    'total_watch_time_minutes'
]

In [ ]:
user_metrics = user_metrics.merge(
    total_watch_time,
    on='profile_name',
    how='left'
)

In [ ]:
user_metrics.head()

## Engagement Segmentation

In [ ]:
def engagement_level(x):

    if x < 1:
        return 'Low'

    elif x < 3:
        return 'Medium'

    else:
        return 'High'

In [ ]:
user_metrics['engagement_segment'] = (
    user_metrics['avg_watch_per_day']
    .apply(engagement_level)
)

In [ ]:
user_metrics

In [ ]:
def engagement_level(x):

    if x < 3:
        return 'Low'

    elif x < 7:
        return 'Medium'

    else:
        return 'High'

In [ ]:
user_metrics['engagement_segment'] = (
    user_metrics['avg_watch_per_day']
    .apply(engagement_level)
)

In [ ]:
user_metrics[['profile_name',
              'avg_watch_per_day',
              'engagement_segment']]

# 4. Churn Analysis

In [ ]:
latest_activity = (
    view_df.groupby('Profile Name')['Start Time']
    .max()
    .reset_index()
)

In [ ]:
latest_activity.columns = [
    'profile_name',
    'last_activity_date'
]

In [ ]:
latest_activity

In [ ]:
dataset_last_date = view_df['Start Time'].max()

dataset_last_date

## Days Since Last Activity

In [ ]:
latest_activity['days_since_last_activity'] = (

    dataset_last_date
    - latest_activity['last_activity_date']

).dt.days

In [ ]:
latest_activity

In [ ]:
latest_activity['churn'] = (
    latest_activity['days_since_last_activity'] > 30
).astype(int)

In [ ]:
latest_activity

In [ ]:
user_metrics = user_metrics.merge(
    latest_activity,
    on='profile_name',
    how='left'
)

In [ ]:
user_metrics

In [ ]:
user_metrics['churn'].value_counts()

In [ ]:
churn_rate = (
    user_metrics['churn'].mean()
) * 100

churn_rate

# 5. Churn vs Engagement Analysis

In [ ]:
user_metrics.groupby('churn')[[
    
    'total_watch_count',
    'active_days',
    'avg_watch_per_day',
    'total_watch_time_minutes'

]].mean()

In [ ]:
pd.crosstab(
    user_metrics['engagement_segment'],
    user_metrics['churn']
)

## Churn Visualization

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    data=user_metrics,
    x='churn'
)

plt.title("Churn Distribution")

plt.savefig(
    "churn_distribution.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=user_metrics,
    x='churn',
    y='avg_watch_per_day'
)

plt.title("Average Watch Per Day vs Churn")

plt.savefig(
    "avg_watch_vs_churn.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=user_metrics,
    x='churn',
    y='total_watch_time_minutes'
)

plt.title("Total Watch Time vs Churn")

plt.savefig(
    "watch_time_vs_churn.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Outlier-Aware Visualization

In [ ]:
filtered_users = user_metrics[
    user_metrics['total_watch_time_minutes'] < 40000
]

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=filtered_users,
    x='churn',
    y='total_watch_time_minutes'
)

plt.title("Total Watch Time vs Churn (Filtered)")

plt.savefig(
    "filtered_watch_time_vs_churn.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 6. Retention and Cohort Analysis

In [ ]:
first_activity = (
    view_df.groupby('Profile Name')['date']
    .min()
    .reset_index()
)

In [ ]:
first_activity.columns = [
    'Profile Name',
    'first_activity_date'
]

In [ ]:
view_df = view_df.merge(
    first_activity,
    on='Profile Name',
    how='left'
)

In [ ]:
view_df.head()

In [ ]:
view_df['cohort_day'] = (

    pd.to_datetime(view_df['date'])
    - pd.to_datetime(view_df['first_activity_date'])

).dt.days

In [ ]:
view_df[['Profile Name',
         'date',
         'first_activity_date',
         'cohort_day']].head()

In [ ]:
retention = (
    view_df.groupby(['first_activity_date', 'cohort_day'])[
        'Profile Name'
    ]
    .nunique()
    .reset_index()
)

In [ ]:
retention.columns = [
    'cohort_date',
    'cohort_day',
    'active_users'
]

In [ ]:
retention.head()

In [ ]:
retention_pivot = retention.pivot_table(
    index='cohort_date',
    columns='cohort_day',
    values='active_users'
)

In [ ]:
retention_pivot

In [ ]:
plt.figure(figsize=(14,8))

sns.heatmap(
    retention_pivot,
    annot=True,
    fmt='.0f'
)

plt.title("Retention Cohort Analysis")

plt.savefig(
    "retention_cohort_heatmap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
cohort_sizes = retention_pivot.iloc[:,0]

In [ ]:
retention_rate = retention_pivot.divide(
    cohort_sizes,
    axis=0
)

In [ ]:
retention_rate.head()

## Retention Rate Matrix

In [ ]:
plt.figure(figsize=(14,8))

sns.heatmap(
    retention_rate,
    annot=False,
    cmap='Blues'
)

plt.title("Retention Rate Cohort Analysis")

plt.savefig(
    "retention_rate_heatmap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 7. Search Behavior Analysis

In [ ]:
query = "SELECT * FROM search_history"

search_df = pd.read_sql(query, engine)

In [ ]:
search_df.head()

In [ ]:
search_df.shape

In [ ]:
search_df.columns

In [ ]:
search_df.info()

In [ ]:
search_df['Utc Timestamp'] = pd.to_datetime(
    search_df['Utc Timestamp']
)

In [ ]:
search_df.info()

## Most Common Search Queries

In [ ]:
search_df['Query Typed'] \
    .value_counts() \
    .head(10)

In [ ]:
top_queries = (
    search_df['Query Typed']
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10,5))

top_queries.plot(kind='bar')

plt.title("Top 10 Search Queries")

plt.ylabel("Search Count")

plt.savefig(
    "top_search_queries.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
search_df['Profile Name'] \
    .value_counts()

In [ ]:
search_user_counts = (
    search_df['Profile Name']
    .value_counts()
)

plt.figure(figsize=(8,5))

search_user_counts.plot(kind='bar')

plt.title("Search Activity by User")

plt.ylabel("Search Count")

plt.savefig(
    "search_activity_by_user.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
search_df['Action'].value_counts()

In [ ]:
action_counts = (
    search_df['Action']
    .value_counts()
)

plt.figure(figsize=(8,5))

action_counts.plot(kind='bar')

plt.title("Search Action Distribution")

plt.ylabel("Count")

plt.savefig(
    "search_action_distribution.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 8. Search vs Watch Behavior

In [ ]:
search_counts = (
    search_df.groupby('Profile Name')
    .size()
    .reset_index(name='search_count')
)

In [ ]:
search_counts

In [ ]:
comparison_df = user_metrics.merge(
    search_counts,
    left_on='profile_name',
    right_on='Profile Name',
    how='left'
)

In [ ]:
comparison_df['search_count'] = (
    comparison_df['search_count']
    .fillna(0)
)

In [ ]:
comparison_df[[
    'profile_name',
    'total_watch_count',
    'search_count'
]]

In [ ]:
comparison_df['search_to_watch_ratio'] = (

    comparison_df['search_count']
    / comparison_df['total_watch_count']

)

In [ ]:
comparison_df[[
    'profile_name',
    'search_count',
    'total_watch_count',
    'search_to_watch_ratio'
]]

In [ ]:
plt.figure(figsize=(10,5))

sns.scatterplot(
    data=comparison_df,
    x='search_count',
    y='total_watch_count'
)

plt.title("Search Count vs Watch Count")

plt.savefig(
    "search_vs_watch_scatter.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 9. SQL-Based Business Metrics

## Daily Active Users

In [ ]:
query = """

SELECT
    DATE("Start Time") AS activity_date,
    COUNT(DISTINCT "Profile Name") AS dau

FROM viewing_activity

GROUP BY activity_date

ORDER BY activity_date;

"""

In [ ]:
dau_df = pd.read_sql(query, engine)

dau_df.head()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    dau_df['activity_date'],
    dau_df['dau']
)

plt.title("Daily Active Users")

plt.xlabel("Date")

plt.ylabel("DAU")

plt.savefig(
    "daily_active_users.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
query = """

SELECT
    "Profile Name",
    COUNT(*) AS total_views

FROM viewing_activity

GROUP BY "Profile Name"

ORDER BY total_views DESC;

"""

In [ ]:
top_users_sql = pd.read_sql(query, engine)

top_users_sql

# 10. Clickstream Behavior Analysis

In [ ]:
query = "SELECT * FROM clickstream"

click_df = pd.read_sql(query, engine)

In [ ]:
click_df.head()

In [ ]:
click_df.shape

In [ ]:
click_df.columns

In [ ]:
click_df.info()

In [ ]:
click_df['Click Utc Ts'] = pd.to_datetime(
    click_df['Click Utc Ts']
)

In [ ]:
click_df['Navigation Level'] \
    .value_counts()

## Navigation Level Analysis

In [ ]:
nav_counts = (
    click_df['Navigation Level']
    .value_counts()
)

plt.figure(figsize=(8,5))

nav_counts.plot(kind='bar')

plt.title("Navigation Level Distribution")

plt.ylabel("Count")

plt.savefig(
    "navigation_level_distribution.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
click_df['Source'].value_counts()

In [ ]:
source_counts = (
    click_df['Source']
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10,5))

source_counts.plot(kind='bar')

plt.title("Top Traffic Sources")

plt.ylabel("Count")

plt.savefig(
    "traffic_sources.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# 11. Product Insights 

### 1. High inactivity strongly correlates with churn
Users who remain inactive for long periods are significantly more likely to churn.

### 2. Highly engaged users show stronger retention behavior
Users with higher watch counts and longer total watch times tend to remain active on the platform.

### 3. Search activity appears correlated with engagement
Users who search more frequently also tend to consume more content.

### 4. Search interactions do not always convert into viewing behavior
Many users interact with search results, but not every interaction leads to actual viewing activity. This may indicate content discovery friction or recommendation mismatches.

### 5. Heavy users dominate platform activity
A small number of highly engaged users contribute disproportionately to total platform activity.

# Final Conclusion

This project analyzed Netflix user behavior using viewing activity, search history, and clickstream interaction data.

The analysis focused on:
- user engagement,
- churn behavior,
- retention patterns,
- search interactions,
- and navigation activity.

Behavioral feature engineering and SQL-based business metrics were used to generate actionable product insights and retention-focused recommendations.